# Haldane相与SPT

探索对称保护拓扑相的物理。

## 学习目标

1. 理解Haldane猜想
2. 验证边缘态四重简并
3. 计算弦序参数
4. 研究对称性保护

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../../common')

from utils.tensor_utils import spin_operators

%matplotlib inline

## 1. Haldane猜想

### 物理图像

- **半整数自旋**: 无能隙，临界态
- **整数自旋**: 有能隙，Haldane相

### S=1/2 vs S=1

海森堡模型: $H = J \sum_i \mathbf{S}_i \cdot \mathbf{S}_{i+1}$

In [ ]:
# 比较S=1/2和S=1的能谱

def plot_energy_gap_comparison():
    """
    比较不同自旋的能隙
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # S=1/2: 无能隙(模拟数据)
    L_values = np.arange(4, 21, 2)
    gap_half = 1.0 / L_values  # 有限尺寸标度: Δ ~ 1/L
    
    ax1.plot(L_values, gap_half, 'o-', markersize=8, linewidth=2, color='blue')
    ax1.set_xlabel('System Size L', fontsize=12)
    ax1.set_ylabel('Energy Gap Δ', fontsize=12)
    ax1.set_title('S=1/2 Heisenberg Chain (Gapless)', fontsize=13)
    ax1.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    
    # S=1: 有能隙
    gap_one = 0.41 * np.ones_like(L_values) + 0.1 * np.exp(-L_values / 5)
    
    ax2.plot(L_values, gap_one, 's-', markersize=8, linewidth=2, color='green')
    ax2.set_xlabel('System Size L', fontsize=12)
    ax2.set_ylabel('Energy Gap Δ', fontsize=12)
    ax2.set_title('S=1 Haldane Chain (Gapped)', fontsize=13)
    ax2.axhline(0.41, color='red', linestyle='--', alpha=0.5, 
               label='Thermodynamic Limit')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Haldane猜想:")
    print("  S=1/2: Δ → 0 (无能隙)")
    print("  S=1:   Δ → 0.41J (有能隙)")

plot_energy_gap_comparison()

## 2. AKLT态的VBS图像

### 价键固体(Valence Bond Solid)

每个S=1分解为两个S=1/2:

```
Site i:    ●—●
Site i+1:  ●—●  
            ╲╱    ← 单态键
```

In [ ]:
# VBS态的MPS表示

def aklt_mps_tensor():
    """
    AKLT态的精确MPS张量
    
    返回: A[s] 其中s=0,1,2对应Sz=-1,0,+1
    """
    # 键维度 χ=2 (两个自旋-1/2)
    A = np.zeros((2, 2, 3), dtype=complex)
    
    # 精确形式(归一化)
    sqrt2 = np.sqrt(2)
    
    # Sz = -1: |↓↓⟩
    A[:, :, 0] = [[0, 0],
                  [1, 0]]
    
    # Sz = 0: (|↑↓⟩ + |↓↑⟩)/√2
    A[:, :, 1] = [[0, 1/sqrt2],
                  [1/sqrt2, 0]]
    
    # Sz = +1: |↑↑⟩
    A[:, :, 2] = [[0, 1],
                  [0, 0]]
    
    return A

A_aklt = aklt_mps_tensor()

print("AKLT MPS张量:")
print(f"形状: {A_aklt.shape} (χ_left=2, χ_right=2, d=3)")
print("\n这是精确的基态！")

# 可视化张量
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for s, label in enumerate([r'$S^z=-1$', r'$S^z=0$', r'$S^z=+1$']):
    im = axes[s].imshow(np.abs(A_aklt[:, :, s]), cmap='YlOrRd', vmin=0, vmax=1)
    axes[s].set_title(label, fontsize=13)
    axes[s].set_xlabel('Right Index', fontsize=11)
    axes[s].set_ylabel('Left Index', fontsize=11)
    plt.colorbar(im, ax=axes[s])

plt.tight_layout()
plt.show()

## 3. 边缘态

### 开放边界的四重简并

两个边缘自旋-1/2组合:
- 三重态: S=1, Sz = -1, 0, +1
- 单态: S=0, Sz = 0

In [ ]:
# 模拟边缘态能级

def plot_edge_state_spectrum():
    """
    绘制边缘态能谱
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # PBC: 唯一基态
    E_pbc = [0]  # 基态
    E_excited_pbc = [0.41, 0.82, 1.2]  # 激发态
    
    ax1.scatter([1], E_pbc, s=200, c='blue', marker='o', label='Ground State')
    ax1.scatter([1, 1, 1], E_excited_pbc, s=100, c='red', marker='s', 
               alpha=0.6, label='Excited States')
    ax1.set_xlim(0.5, 1.5)
    ax1.set_ylim(-0.2, 1.5)
    ax1.set_xticks([1])
    ax1.set_xticklabels(['PBC'])
    ax1.set_ylabel('Energy (J)', fontsize=12)
    ax1.set_title('Periodic Boundary (Unique Ground State)', fontsize=13)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # OBC: 四重简并
    E_obc_gs = [0, 0.001, 0.002, 0.003]  # 几乎简并(有限尺寸分裂)
    labels = ['Triplet\n$S_z=+1$', 'Triplet\n$S_z=0$', 
             'Triplet\n$S_z=-1$', 'Singlet\n$S=0$']
    
    colors = ['red', 'orange', 'red', 'blue']
    for i, (E, label, color) in enumerate(zip(E_obc_gs, labels, colors)):
        ax2.scatter([i+1], [E], s=200, c=color, marker='o', alpha=0.7)
        ax2.text(i+1, E-0.05, label, ha='center', fontsize=9)
    
    ax2.set_xlim(0.5, 4.5)
    ax2.set_ylim(-0.3, 0.5)
    ax2.set_xticks(range(1, 5))
    ax2.set_xticklabels(['', '', '', ''])
    ax2.set_ylabel('Energy (J)', fontsize=12)
    ax2.set_title('Open Boundary (4-fold Degeneracy)', fontsize=13)
    ax2.axhline(0, color='green', linestyle='--', linewidth=2, 
               alpha=0.5, label='Ground State Manifold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("边缘态分析:")
    print("  PBC: 1个基态(无边缘)")
    print("  OBC: 4个简并基态(两个边缘自旋-1/2)")
    print("  有限尺寸分裂: ΔE ~ exp(-L/ξ)")

plot_edge_state_spectrum()

## 4. 弦序参数

### 定义

$$
O_z(i,j) = \langle S_i^z \exp\left(i\pi \sum_{k=i+1}^{j-1} S_k^z\right) S_j^z \rangle
$$

### AKLT态的精确值

$$
O_z^{\text{AKLT}} = -\frac{4}{9} \approx 0.444
$$

（数值修正: ~0.374）

In [ ]:
# 弦序参数vs距离

def plot_string_order_analysis():
    """
    分析弦序参数的行为
    """
    distances = np.arange(1, 25)
    
    # Haldane相: 长程序
    O_haldane = 0.374 * np.ones_like(distances, dtype=float) + \
                0.02 * np.exp(-distances / 10)
    
    # 平凡相: 指数衰减
    O_trivial = 0.4 * np.exp(-distances / 3)
    
    # 普通关联函数（对比）
    C_normal = 0.5 * np.exp(-distances / 2) * np.cos(np.pi * distances)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 弦序
    ax1.plot(distances, O_haldane, 'o-', label='Haldane Phase', 
            markersize=6, linewidth=2)
    ax1.plot(distances, O_trivial, 's-', label='Trivial Phase',
            markersize=6, linewidth=2)
    ax1.axhline(0.374, color='red', linestyle='--', alpha=0.5,
               label='AKLT Theory')
    ax1.set_xlabel('Distance $r$', fontsize=12)
    ax1.set_ylabel('String Order $O_z(r)$', fontsize=12)
    ax1.set_title('String Order Parameter', fontsize=13)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 对比普通关联
    ax2.plot(distances, np.abs(O_haldane), 'o-', 
            label='String Order (Haldane)', markersize=6, linewidth=2)
    ax2.plot(distances, np.abs(C_normal), '^-', 
            label='Normal Correlation', markersize=6, linewidth=2)
    ax2.set_xlabel('Distance $r$', fontsize=12)
    ax2.set_ylabel('Absolute Value', fontsize=12)
    ax2.set_title('String Order vs Normal Correlation', fontsize=13)
    ax2.set_yscale('log')
    ax2.legend()
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print("弦序参数特征:")
    print("  Haldane相: 长程弦序 O_z → const")
    print("  平凡相: 指数衰减 O_z ~ exp(-r/ξ)")
    print("  普通关联: 总是指数衰减")
    print("\n这是隐藏的拓扑序！")

plot_string_order_analysis()

## 5. 对称性破缺效应

### 添加单离子各向异性

$$
H' = H_{\text{AKLT}} + D \sum_i (S_i^z)^2
$$

当D足够大时,破坏SO(3)对称性 → 边缘态消失！

In [ ]:
# 对称破缺场的影响

def plot_symmetry_breaking_effect():
    """
    研究对称破缺场对边缘态的影响
    """
    D_values = np.linspace(0, 2, 50)
    
    # 能级分裂(模拟)
    # D=0: 简并; D大: 分裂
    splitting_triplet = D_values**2 / (1 + D_values)
    splitting_singlet = 0.5 * D_values
    
    # 弦序参数
    string_order = 0.374 * np.exp(-D_values / 0.5)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 能级分裂
    ax1.plot(D_values, splitting_triplet, '-', linewidth=2,
            label='Triplet Splitting')
    ax1.plot(D_values, splitting_singlet, '--', linewidth=2,
            label='Singlet Energy Shift')
    ax1.set_xlabel('Anisotropy $D/J$', fontsize=12)
    ax1.set_ylabel('Energy Splitting (J)', fontsize=12)
    ax1.set_title('Edge State Splitting', fontsize=13)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 弦序衰减
    ax2.plot(D_values, string_order, 'o-', linewidth=2, markersize=4)
    ax2.axhline(0.374, color='red', linestyle='--', alpha=0.5,
               label='Pure Haldane')
    ax2.set_xlabel('Anisotropy $D/J$', fontsize=12)
    ax2.set_ylabel('String Order $O_z$', fontsize=12)
    ax2.set_title('String Order vs Anisotropy', fontsize=13)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("对称性保护:")
    print("  D=0: SO(3)对称,边缘态简并")
    print("  D≠0: 破缺对称,简并解除")
    print("  D→∞: 边缘态消失")

plot_symmetry_breaking_effect()

## 练习

1. 验证AKLT MPS张量满足归一化条件
2. 计算边缘纠缠熵 $S_{\text{edge}} = \ln 2$
3. 研究有限尺寸下的边缘态分裂
4. 探索S=2的Haldane链

## 扩展阅读

- Haldane (1983) - 原始猜想
- AKLT (1987) - 精确可解模型  
- Chen, Gu & Wen (2011) - SPT分类
- Pollmann & Turner (2012) - 数值诊断